In [1]:
import numpy as np # For numerical operations
import torch # For PyTorch operations
import torch.nn as nn # For neural network layers
import torch.nn.functional as F # For activation functions and other functional operations
import torchvision.transforms as T # For image transformations
import torchvision.datasets as datasets # For standard datasets like CIFAR-10
from torch.utils.data import DataLoader # For creating data loaders

In [2]:
# Setting up the test dataset and data loader for a simple CNN using CIFAR-10

from pathlib import Path
from torchvision.datasets.utils import check_integrity, extract_archive

data_dir = './data'
cifar_test_batch = Path(data_dir) / 'cifar-10-batches-py' / 'test_batch'
if not cifar_test_batch.is_file():
    cifar_archive = Path(data_dir) / datasets.CIFAR10.filename
    if not cifar_archive.is_file():
        raise FileNotFoundError(
            f"Place cifar-10-python.tar.gz in {Path(data_dir).resolve()} and rerun this cell."
        )
    if not check_integrity(cifar_archive, datasets.CIFAR10.tgz_md5):
        raise ValueError(f"CIFAR-10 archive checksum mismatch: {cifar_archive.resolve()}")
    print(f"Extracting local CIFAR-10 archive: {cifar_archive.resolve()}")
    extract_archive(str(cifar_archive), data_dir)

transform = T.Compose([
    T.Resize(224),
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
])

test_dataset  = datasets.CIFAR10(root=data_dir, train=False, download=False, transform=transform)
test_loader  = DataLoader(
        test_dataset,  
        batch_size=64, 
        shuffle=False
    )
print(f"Loaded {len(test_dataset):,} CIFAR-10 test images from {Path(data_dir).resolve()}")

Loaded 10,000 CIFAR-10 test images from C:\Users\nomadant\AI_Engineering_Lessons\Codecademy\Neural Network Architectures\data


In [3]:
import torchvision.models as M # For pre-defined models like ResNet-18

# Move to CPU/GPU device -- DO NOT MODIFY
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load ResNet18 and adapt for CIFAR-10 
model_resnet18 = M.resnet18(weights=None)
in_feats = model_resnet18.fc.in_features
model_resnet18.fc = nn.Linear(in_feats, 10)

# Load pre-trained weights fine-tuned on CIFAR-10
state_dict_resnet18 = torch.load("models/resnet18_cifar10.pt", map_location="cpu", weights_only=True)
model_resnet18.load_state_dict(state_dict_resnet18)

# Move to device and set to evaluation mode
model_resnet18 = model_resnet18.to(device)
model_resnet18.eval()

# Show output
from custom_torchinfo import custom_summary
custom_summary(model_resnet18, input_size=(1, 3, 224, 224))

Layer (type)             Output Shape             Param #        
Conv2d                   [1, 64, 112, 112]        9408           
BatchNorm2d              [1, 64, 112, 112]        128            
ReLU                     [1, 64, 112, 112]        0              
MaxPool2d                [1, 64, 56, 56]          0              
Conv2d                   [1, 64, 56, 56]          36864          
BatchNorm2d              [1, 64, 56, 56]          128            
ReLU                     [1, 64, 56, 56]          0              
Conv2d                   [1, 64, 56, 56]          36864          
BatchNorm2d              [1, 64, 56, 56]          128            
ReLU                     [1, 64, 56, 56]          0              
BasicBlock               [1, 64, 56, 56]          73984          
Conv2d                   [1, 64, 56, 56]          36864          
BatchNorm2d              [1, 64, 56, 56]          128            
ReLU                     [1, 64, 56, 56]          0              
Conv2d    

In [4]:
# Function to generate predictions for a given model and dataloader

def predict(model, dataloader, device="cpu"):
    model.eval()
    all_logits = []
    all_predictions = []
    all_labels = []

    with torch.inference_mode():
        for batch_X, batch_y in dataloader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            # Forward pass
            logits = model(batch_X)

            # Convert logits to predicted labels
            probs = torch.softmax(logits, dim=1)
            preds = probs.argmax(dim=1)

            # Save predictions
            all_logits.append(logits.cpu().numpy())
            all_predictions.append(preds.cpu().numpy())
            all_labels.append(batch_y.cpu().numpy())

    # Join predictions
    y_logits = np.concatenate(all_logits, axis=0)
    y_pred   = np.concatenate(all_predictions, axis=0)
    y_true   = np.concatenate(all_labels, axis=0)

    return y_logits, y_pred, y_true

# Generate test predictions
y_logits_resnet18, y_pred_resnet18, y_true = predict(model_resnet18, test_loader, device=device)

# Show output
print("First 10 predictions:", y_pred_resnet18[:10])
print("First 10 labels:", y_true[:10])

First 10 predictions: [3 8 1 0 6 6 1 6 3 1]
First 10 labels: [3 8 8 0 6 6 1 6 3 1]


In [5]:
# Evaluating the ResNet-18 model on the test dataset

from sklearn.metrics import accuracy_score, classification_report

test_accuracy_resnet18 = accuracy_score(y_true, y_pred_resnet18)
report_resnet18 = classification_report(y_true, y_pred_resnet18, labels=range(10), target_names=test_dataset.classes, zero_division=0)

# Show output
print(f"Accuracy: {test_accuracy_resnet18:.4f}")
print(report_resnet18)

Accuracy: 0.8853
              precision    recall  f1-score   support

    airplane       0.83      0.94      0.88      1000
  automobile       0.98      0.88      0.93      1000
        bird       0.87      0.84      0.85      1000
         cat       0.83      0.72      0.77      1000
        deer       0.90      0.88      0.89      1000
         dog       0.83      0.85      0.84      1000
        frog       0.89      0.94      0.91      1000
       horse       0.90      0.94      0.92      1000
        ship       0.95      0.91      0.93      1000
       truck       0.89      0.95      0.92      1000

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.88     10000
weighted avg       0.89      0.89      0.88     10000



In [ ]:
# Loading and modifying the MobileNet model for CIFAR-10 classification

import torchvision.models as M

# Load the MobileNet model and modify it for CIFAR-10 classification
model_mobilenet = M.mobilenet_v3_small(weights=None)
in_feats = model_mobilenet.classifier[-1].in_features
model_mobilenet.classifier[-1] = torch.nn.Linear(in_feats, 10)  # type: ignore , Assuming 10 classes

# Load the state dictionary and move the model to the appropriate device
state_dict = torch.load("./models/mobilenet_cifar10.pt", map_location="cpu", weights_only=True)
model_mobilenet.load_state_dict(state_dict)
model_mobilenet = model_mobilenet.to(device)
model_mobilenet.eval()

# Display the model summary using custom_torchinfo
from custom_torchinfo import custom_summary
custom_summary(model_mobilenet, input_size=(1, 3, 224, 224))

# Generate predictions and evaluate the MobileNet model on the test dataset
y_logits_mobilenet, y_pred_mobilenet, y_true = predict(model_mobilenet, test_loader, device=device)
test_accuracy_mobilenet = accuracy_score(y_true, y_pred_mobilenet)
report_mobilenet = classification_report(y_true, y_pred_mobilenet, labels=range(10), target_names=test_dataset.classes, zero_division=0)

# Show output
print(f"Accuracy: {test_accuracy_mobilenet:.4f}")
print(report_mobilenet)

Layer (type)             Output Shape             Param #        
Conv2d                   [1, 16, 112, 112]        432            
BatchNorm2d              [1, 16, 112, 112]        32             
Hardswish                [1, 16, 112, 112]        0              
Conv2d                   [1, 16, 56, 56]          144            
BatchNorm2d              [1, 16, 56, 56]          32             
ReLU                     [1, 16, 56, 56]          0              
AdaptiveAvgPool2d        [1, 16, 1, 1]            0              
Conv2d                   [1, 8, 1, 1]             136            
ReLU                     [1, 8, 1, 1]             0              
Conv2d                   [1, 16, 1, 1]            144            
Hardsigmoid              [1, 16, 1, 1]            0              
SqueezeExcitation        [1, 16, 56, 56]          280            
Conv2d                   [1, 16, 56, 56]          256            
BatchNorm2d              [1, 16, 56, 56]          32             
InvertedRe

In [7]:
import time

# Grab the first image in the testing dataset for latency and memory measurements

batch_X, batch_y = next(iter(test_loader))
x1 = batch_X[0].unsqueeze(0).to(device)

def measure_latency(model, x, iters=30):
    """Return average latency (ms) per forward pass."""
    model.eval()
    with torch.inference_mode():
        for _ in range(5):
            model(x)
    if x.device.type == "cuda":
        torch.cuda.synchronize(x.device)
    start = time.perf_counter()
    with torch.inference_mode():
        for _ in range(iters):
            _ = model(x)
            if x.device.type == "cuda":
                torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    return (elapsed / iters) * 1e3  # ms

def measure_gpu_memory(model, x):
    """Return current and peak GPU memory in MB after one forward."""
    if x.device.type != "cuda":
        return None
    torch.cuda.synchronize(x.device)
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(x.device)
    
    model.eval()
    with torch.inference_mode():
        y = model(x) # Optional - delete predictions afterwards
        
    torch.cuda.synchronize(x.device)
    current = torch.cuda.memory_allocated(x.device) / (1024**2)
    peak    = torch.cuda.max_memory_allocated(x.device) / (1024**2)
    return current, peak

# Measure latency
latency_mobilenet = measure_latency(model_mobilenet, x1, iters=30)
latency_resnet18 = measure_latency(model_resnet18, x1, iters=30)

# Measure GPU allocation
memory_mobilenet = measure_gpu_memory(model_mobilenet, x1)
memory_resnet18 = measure_gpu_memory(model_resnet18, x1)

# Show output
print("Latency Comparison")
print("MobileNetV3-Small:", f"{latency_mobilenet:.4f} ms")
print("ResNet-18:",         f"{latency_resnet18:.4f} ms")
print("====================================================================")

print("Memory Usage Comparison")
if memory_mobilenet is None or memory_resnet18 is None:
    print("GPU memory measurements are unavailable on CPU.")
else:
    print("GPU allocations include both loaded models, not isolated model footprints.")
    print(f"MobileNetV3-Small: [GPU Memory] Current allocated: {memory_mobilenet[0]:.2f} MB | Peak during forward: {memory_mobilenet[1]:.2f} MB")
    print(f"ResNet-18: [GPU Memory] Current allocated: {memory_resnet18[0]:.2f} MB | Peak during forward: {memory_resnet18[1]:.2f} MB")
print("====================================================================")

print("Classification Performance Comparison")
print("MobileNetV3-Small Performance:")
print(f"Accuracy: {test_accuracy_mobilenet:.4f}")
print(report_mobilenet)
print("ResNet-18 Performance:")
print(f"Accuracy: {test_accuracy_resnet18:.4f}")
print(report_resnet18)

Latency Comparison
MobileNetV3-Small: 9.5078 ms
ResNet-18: 3.8131 ms
Memory Usage Comparison
GPU allocations include both loaded models, not isolated model footprints.
MobileNetV3-Small: [GPU Memory] Current allocated: 57.97 MB | Peak during forward: 59.88 MB
ResNet-18: [GPU Memory] Current allocated: 57.97 MB | Peak during forward: 84.44 MB
Classification Performance Comparison
MobileNetV3-Small Performance:
Accuracy: 0.8492
              precision    recall  f1-score   support

    airplane       0.79      0.92      0.85      1000
  automobile       0.89      0.97      0.93      1000
        bird       0.86      0.75      0.80      1000
         cat       0.81      0.63      0.71      1000
        deer       0.90      0.83      0.86      1000
         dog       0.80      0.78      0.79      1000
        frog       0.78      0.94      0.85      1000
       horse       0.93      0.87      0.90      1000
        ship       0.81      0.95      0.87      1000
       truck       0.97      

In [8]:
# Cleaning up GPU memory by moving models to CPU and deleting them

import gc, torch # For garbage collection and PyTorch operations

model_mobilenet.to("cpu")
model_resnet18.to("cpu")

del model_mobilenet, model_resnet18, state_dict, state_dict_resnet18, x1
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()